In [1]:
context_mode = "structured" # or "structured", templated
RANDOM_STATE = 42

In [2]:
import os
import sys

# Get the working directory and add project root to path
working_dir = '/Users/vuong/Downloads/research/repos/ClinicNumRobBench'
os.chdir(working_dir)
sys.path.append(working_dir)

print(f"Current working directory: {os.getcwd()}")
print(f"Project root added to path: {working_dir}")

Current working directory: /Users/vuong/Downloads/research/repos/ClinicNumRobBench
Project root added to path: /Users/vuong/Downloads/research/repos/ClinicNumRobBench


# Problem Building

In [3]:
import pandas as pd

out_sample_path = "data/mimiciv/mimic4ed/200_sampled.csv"

In [4]:
df = pd.read_csv(out_sample_path)
df['charttime'] = pd.to_datetime(df['charttime'])
df

,subject_id,stay_id,charttime,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,gender,age,age_cate,vitals_count,vitals_count_category
0,10092020,37333689,2135-06-15 11:11:00,36.4,63,15,100,138,80,6,M,69,older_adult,1,1-5
1,10168562,37598096,2124-07-24 18:53:00,36.9,62,16,99,122,65,4,F,18,pediatric,1,1-5
2,10197727,39726422,2158-06-06 20:22:00,36.7,95,16,99,160,72,3,M,40,young_adult,3,1-5
3,10197727,39726422,2158-06-06 21:43:00,39.0,112,18,98,172,79,5,M,40,young_adult,3,1-5
4,10197727,39726422,2158-06-07 00:10:00,38.3,122,24,96,163,61,8,M,40,young_adult,3,1-5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
832,19985757,37571161,2174-09-01 11:34:00,37.0,82,18,100,124,69,8,F,77,older_adult,2,1-5
833,19986107,32928247,2171-06-23 19:24:00,36.6,101,20,99,149,79,5,F,55,middle_age,2,1-5
834,19986107,32928247,2171-06-23 23:07:00,36.2,110,22,95,183,95,7,F,55,middle_age,2,1-5
835,19990786,34314807,2157-07-12 14:51:00,36.9,65,18,95,92,42,6,M,80,older_adult,2,1-5


In [5]:
df.groupby('subject_id').last().groupby(['vitals_count_category','gender', 'age_cate']).size()

vitals_count_category  gender  age_cate   
1-5                    F       elderly        12
                               middle_age     22
                               older_adult    30
                               pediatric       1
                               young_adult    10
                       M       elderly         8
                               middle_age     30
                               older_adult    38
                               pediatric       1
                               young_adult    12
20-50                  F       elderly         1
                               middle_age      1
                               older_adult     1
                               young_adult     1
                       M       elderly         1
                               middle_age      1
                               older_adult     1
                               young_adult     1
5-20                   F       elderly         2
                          

In [6]:
df['subject_id'].nunique()

206

## Verbalize subject context

By group subject, verbalize into context with template

In [7]:
df.columns

Index(['subject_id', 'stay_id', 'charttime', 'temperature', 'heartrate',
       'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'gender', 'age', 'age_cate',
       'vitals_count', 'vitals_count_category'],
      dtype='str')

In [8]:
df.drop(columns=["stay_id"], inplace=True)
# df.rename(columns={
#     "subject_id":"patient",
#     "date":"recorded on",
#     "time":"recorded at",
#     "temperature":"temperature",
#     "heartrate":"heart rate",
#     "resprate":"respiratory rate",
#     "o2sat":"O2 saturation",
#     "sbp":"systolic blood pressure",
#     "dbp":"diastolic blood pressure",
    
# }, inplace=True)

### Choose context mode & update Output_dir

In [9]:
output_dir = os.path.join("data", context_mode)
print("Data output: ", output_dir)

Data output:  data/structured


In [10]:
def format_df(df):
    return df.apply(lambda r: {
        "question": "\n\n".join([r['context'], r['question']]),
        "answer": r['answer'],
        "open_ended_answer": r['answer'],
        "dataset_source": r['sub_type']
    }, axis=1).apply(pd.Series)

### Apply

In [11]:
import textwrap

df['gender'] = df['gender'].map(lambda x: "male" if x == "M" else "female") # because just have M and F in the data
df['possessive_pronoun'] = df['gender'].map(lambda x: "His" if x == "male" else "Her")
if context_mode == "templated":
    standard_template = "At {charttime}, the {age}-year-old {gender} reports pain level is {pain}. {possessive_pronoun} vital sign at the time is temperature {temperature}°C, heart rate {heartrate} bpm, respiratory rate {resprate} bpm, O2 saturation {o2sat}%, blood pressure {sbp}/{dbp} mmHg."
elif context_mode == "structured":
    standard_template = textwrap.dedent("""{{
    chart_time: {charttime},
    age: {age},
    gender: {gender},
    pain: {pain},
    temperature: {temperature},
    heart_rate: {heartrate},
    resp_rate: {resprate},
    o2_sat: {o2sat},
    bp: {sbp}/{dbp}
}}""")
else:
    raise ValueError(f"Invalid context model: {context_mode}")
df['standard_verbal'] = df.apply(lambda row: standard_template.format(**row), axis=1)
df[['subject_id', 'standard_verbal']]

,subject_id,standard_verbal
0,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a..."
1,10168562,"{\n chart_time: 2124-07-24 18:53:00,\n a..."
2,10197727,"{\n chart_time: 2158-06-06 20:22:00,\n a..."
3,10197727,"{\n chart_time: 2158-06-06 21:43:00,\n a..."
4,10197727,"{\n chart_time: 2158-06-07 00:10:00,\n a..."
...,...,...
832,19985757,"{\n chart_time: 2174-09-01 11:34:00,\n a..."
833,19986107,"{\n chart_time: 2171-06-23 19:24:00,\n a..."
834,19986107,"{\n chart_time: 2171-06-23 23:07:00,\n a..."
835,19990786,"{\n chart_time: 2157-07-12 14:51:00,\n a..."


### grouping

In [12]:
# Group by subject_id and merge standard_verbal with "\n\n"
# short by record time
if context_mode == "templated":
    split_char = "\n"
elif context_mode == "structured":
    split_char = ",\n"
else:
    raise ValueError(f"Invalid context model: {context_mode}")
df_verbal = df.sort_values(['subject_id', 'charttime']).reset_index(drop=True).groupby('subject_id')['standard_verbal'].apply(lambda x: split_char.join(x)).reset_index()
# Replace age-gender patterns with pronouns
# Replace age-gender patterns with pronouns for the first occurrence, then use pronouns for subsequent occurrences
def replace_age_gender_with_pronouns(text, split_char):
    lines = text.split(split_char)
    if len(lines) <= 1:
        return text
    
    # Keep the first line as is, replace in subsequent lines
    result_lines = [lines[0]]
    for line in lines[1:]:
        line = line.replace(r'the \d+-year-old female', 'she', 1) if 'female' in line else line
        line = line.replace(r'the \d+-year-old male', 'he', 1) if 'male' in line else line
        # Use regex for more precise replacement
        import re
        line = re.sub(r'the \d+-year-old female', 'she', line)
        line = re.sub(r'the \d+-year-old male', 'he', line)
        result_lines.append(line)
    
    return split_char.join(result_lines)

df_verbal['standard_verbal'] = df_verbal['standard_verbal'].apply(lambda t: replace_age_gender_with_pronouns(t, split_char.replace(",","")))
df_verbal.head(2)

,subject_id,standard_verbal
0,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a..."
1,10168562,"{\n chart_time: 2124-07-24 18:53:00,\n a..."


In [13]:
print(df_verbal['standard_verbal'][20])

{
    chart_time: 2149-04-15 18:22:00,
    age: 18,
    gender: female,
    pain: 5,
    temperature: 37.7,
    heart_rate: 93,
    resp_rate: 18,
    o2_sat: 97,
    bp: 119/73
},
{
    chart_time: 2149-04-15 20:00:00,
    age: 18,
    gender: female,
    pain: 4,
    temperature: 37.2,
    heart_rate: 90,
    resp_rate: 18,
    o2_sat: 97,
    bp: 110/70
},
{
    chart_time: 2149-04-15 22:53:00,
    age: 18,
    gender: female,
    pain: 0/10,
    temperature: 37.7,
    heart_rate: 81,
    resp_rate: 16,
    o2_sat: 100,
    bp: 111/72
},
{
    chart_time: 2149-04-16 02:36:00,
    age: 18,
    gender: female,
    pain: 4/10,
    temperature: 39.3,
    heart_rate: 101,
    resp_rate: 18,
    o2_sat: 98,
    bp: 107/51
},
{
    chart_time: 2149-04-16 06:39:00,
    age: 18,
    gender: female,
    pain: 0/10,
    temperature: 36.8,
    heart_rate: 70,
    resp_rate: 14,
    o2_sat: 99,
    bp: 99/56
},
{
    chart_time: 2149-04-16 19:49:00,
    age: 18,
    gender: female,
    pain: 4,


In [14]:
# Add vitals_count, gender, age columns to df_grouped based on subject_id
# Get the first occurrence of each subject_id to extract these columns
subject_info = df.groupby('subject_id')[['vitals_count', 'gender', 'age', 'age_cate']].first().reset_index()
df_verbal = df_verbal.merge(subject_info, on='subject_id', how='left')
df_verbal.head(2)


,subject_id,standard_verbal,vitals_count,gender,age,age_cate
0,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a...",1,male,69,older_adult
1,10168562,"{\n chart_time: 2124-07-24 18:53:00,\n a...",1,female,18,pediatric


In [15]:
df[df['subject_id'].isin(df_verbal.head(2)['subject_id'])].drop_duplicates(subset=['subject_id'])

,subject_id,charttime,temperature,heartrate,resprate,o2sat,sbp,dbp,pain,gender,age,age_cate,vitals_count,vitals_count_category,possessive_pronoun,standard_verbal
0,10092020,2135-06-15 11:11:00,36.4,63,15,100,138,80,6,male,69,older_adult,1,1-5,His,"{\n chart_time: 2135-06-15 11:11:00,\n a..."
1,10168562,2124-07-24 18:53:00,36.9,62,16,99,122,65,4,female,18,pediatric,1,1-5,Her,"{\n chart_time: 2124-07-24 18:53:00,\n a..."


# Template DF

In [16]:
template_df = pd.DataFrame(columns=["subject_id","context", "question", "answer", "answer_index","type", "sub_type"])
template_df

,subject_id,context,question,answer,answer_index,type,sub_type


In [17]:
preprocess_mimic4ed_output_dir = "data/mimic4ed"

## Retrieval

In [18]:
import os

In [19]:
retrieval_type = "retrieval"

In [20]:
retrieval_output_dir = os.path.join(preprocess_mimic4ed_output_dir, "retrieval")
formatted_retrieval_output_dir = os.path.join(output_dir, "retrieval")
os.makedirs(retrieval_output_dir, exist_ok=True)
os.makedirs(formatted_retrieval_output_dir, exist_ok=True)

### template1: Direct retrieval

We have 6 number columns: emperature,heartrate,resprate,o2sat,sbp,dbp 

So, I will get from record 1-10 of 2 patients (1 male and 1 female) having 10 records

In [21]:
template1_sub_type = "direct_retrieval"

num_records = [1,10]
num_patients = 1
columns_verbal_mapping = {"temperature":"temperature", 
           "heartrate":"heart rate", 
           "resprate":"respiratory rate", 
           "o2sat":"O2 saturation", 
           "sbp":"systolic blood pressure", 
           "dbp":"diastolic blood pressure"}

question_template = "What was the {column} of the patient at {charttime}?"


template1_retrieval_df = template_df.copy()
for num_record in num_records:
    # select patients with 10 records and 2 patients for each gender
    selected_subjects = df_verbal[df_verbal['vitals_count']==num_record].groupby(["gender", "age_cate"]).apply(lambda x: x.sample(n=num_patients, random_state=RANDOM_STATE))['subject_id'].tolist()
    for i in range(num_record):
        for patient in selected_subjects:
            for column in columns_verbal_mapping:
                sample = df[df['subject_id']==patient].iloc[i]
                question = question_template.format(column=columns_verbal_mapping[column], charttime=sample['charttime'])
                answer = sample[column]
                template1_retrieval_df.loc[len(template1_retrieval_df)] = {
                    "subject_id":patient, 
                    "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
                    "question":question, 
                    "answer":answer, 
                    "answer_index":i,
                    "type":retrieval_type, 
                    "sub_type":template1_sub_type}
            
template1_retrieval_df

,subject_id,context,question,answer,answer_index,type,sub_type
0,11544860,"{\n chart_time: 2174-01-19 09:34:00,\n a...",What was the temperature of the patient at 217...,36.3,0,retrieval,direct_retrieval
1,11544860,"{\n chart_time: 2174-01-19 09:34:00,\n a...",What was the heart rate of the patient at 2174...,90.0,0,retrieval,direct_retrieval
2,11544860,"{\n chart_time: 2174-01-19 09:34:00,\n a...",What was the respiratory rate of the patient a...,24.0,0,retrieval,direct_retrieval
3,11544860,"{\n chart_time: 2174-01-19 09:34:00,\n a...",What was the O2 saturation of the patient at 2...,100.0,0,retrieval,direct_retrieval
4,11544860,"{\n chart_time: 2174-01-19 09:34:00,\n a...",What was the systolic blood pressure of the pa...,113.0,0,retrieval,direct_retrieval
...,...,...,...,...,...,...,...
235,19033304,"{\n chart_time: 2148-12-28 15:08:00,\n a...",What was the heart rate of the patient at 2151...,88.0,9,retrieval,direct_retrieval
236,19033304,"{\n chart_time: 2148-12-28 15:08:00,\n a...",What was the respiratory rate of the patient a...,18.0,9,retrieval,direct_retrieval
237,19033304,"{\n chart_time: 2148-12-28 15:08:00,\n a...",What was the O2 saturation of the patient at 2...,97.0,9,retrieval,direct_retrieval
238,19033304,"{\n chart_time: 2148-12-28 15:08:00,\n a...",What was the systolic blood pressure of the pa...,151.0,9,retrieval,direct_retrieval


#### Save

In [22]:
template1_retrieval_df.to_csv(os.path.join(retrieval_output_dir, "direct_retrieval.csv"), index=False)

In [23]:
formatted_template1_retrieval_df = format_df(template1_retrieval_df)
formatted_template1_retrieval_df.to_csv(os.path.join(formatted_retrieval_output_dir, "direct_retrieval.csv"), index=False)
formatted_template1_retrieval_df

,question,answer,open_ended_answer,dataset_source
0,"{\n chart_time: 2174-01-19 09:34:00,\n a...",36.3,36.3,direct_retrieval
1,"{\n chart_time: 2174-01-19 09:34:00,\n a...",90.0,90.0,direct_retrieval
2,"{\n chart_time: 2174-01-19 09:34:00,\n a...",24.0,24.0,direct_retrieval
3,"{\n chart_time: 2174-01-19 09:34:00,\n a...",100.0,100.0,direct_retrieval
4,"{\n chart_time: 2174-01-19 09:34:00,\n a...",113.0,113.0,direct_retrieval
...,...,...,...,...
235,"{\n chart_time: 2148-12-28 15:08:00,\n a...",88.0,88.0,direct_retrieval
236,"{\n chart_time: 2148-12-28 15:08:00,\n a...",18.0,18.0,direct_retrieval
237,"{\n chart_time: 2148-12-28 15:08:00,\n a...",97.0,97.0,direct_retrieval
238,"{\n chart_time: 2148-12-28 15:08:00,\n a...",151.0,151.0,direct_retrieval


### merge

In [24]:
retrieval_df = template1_retrieval_df
retrieval_df['sub_type'].value_counts()

sub_type
direct_retrieval    240
Name: count, dtype: int64

## Calculation

In [25]:
calculation_type = "calculation"

In [26]:
calculation_output_dir = os.path.join(preprocess_mimic4ed_output_dir, "calculation")
formatted_calculation_output_dir = os.path.join(output_dir, "calculation")
os.makedirs(calculation_output_dir, exist_ok=True)
os.makedirs(formatted_calculation_output_dir, exist_ok=True)

### 1-step

#### Addition - Sum of vital parameters

In [27]:
question_template = "Calculate the sum of systolic and diastolic blood pressure at {charttime}?"
addition_sub_type = "calc_1step_addition"

calc_1step_addition_df = template_df.copy()
# get 50 patients with 5 records each group by vitals_count
vitals_counts = []
for patient in df.groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(5, len(x)), random_state=RANDOM_STATE))['subject_id'].unique():
    patient_records = df[df['subject_id']==patient].reset_index(drop=True)
    patient_records = patient_records.loc[len(patient_records)//3:].sample(n=1)
    question = question_template.format(charttime=str(patient_records['charttime'].values[0]).split('.')[0].replace('T', ' '))
    answer = patient_records['sbp'].values[0] + patient_records['dbp'].values[0]
    calc_1step_addition_df.loc[len(calc_1step_addition_df)] = {
        "subject_id":patient, 
        "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
        "question":question, 
        "answer":answer, 
        "answer_index":patient_records.index.values[0],
        "type":calculation_type, 
        "sub_type":addition_sub_type}
    vitals_counts.append(patient_records['vitals_count'].values[0])

print("vitals_counts: ", vitals_counts)
calc_1step_addition_df.head()

vitals_counts:  [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(7), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(21), np.int64(22), np.int64(23), np.int64(32), np.int64(37), np.int64(40)]


,subject_id,context,question,answer,answer_index,type,sub_type
0,10511623,"{\n chart_time: 2140-10-05 22:40:00,\n a...",Calculate the sum of systolic and diastolic bl...,265,0,calculation,calc_1step_addition
1,18558304,"{\n chart_time: 2177-12-15 10:21:00,\n a...",Calculate the sum of systolic and diastolic bl...,180,0,calculation,calc_1step_addition
2,11227287,"{\n chart_time: 2173-04-13 16:41:00,\n a...",Calculate the sum of systolic and diastolic bl...,261,0,calculation,calc_1step_addition
3,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a...",Calculate the sum of systolic and diastolic bl...,218,0,calculation,calc_1step_addition
4,14411399,"{\n chart_time: 2135-04-02 22:53:00,\n a...",Calculate the sum of systolic and diastolic bl...,166,0,calculation,calc_1step_addition


In [28]:
calc_1step_addition_df['question'][0]

'Calculate the sum of systolic and diastolic blood pressure at 2140-10-05 22:40:00?'

#### Subtraction: Pulse Pressure

In [29]:
subtraction_sub_type = "calc_1step_subtraction"
question_template = "Calculate the pulse pressure at {charttime}?"

calc_1step_subtraction_df = template_df.copy()
vitals_counts = []
for patient in df.groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(5, len(x)), random_state=RANDOM_STATE))['subject_id'].unique():
    patient_records = df[df['subject_id']==patient].reset_index(drop=True)
    patient_records = patient_records.loc[len(patient_records)//3:].sample(n=1)
    question = question_template.format(charttime=str(patient_records['charttime'].values[0]).split('.')[0].replace('T', ' '))
    answer = patient_records['sbp'].values[0] - patient_records['dbp'].values[0]
    calc_1step_subtraction_df.loc[len(calc_1step_subtraction_df)] = {
        "subject_id":patient, 
        "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
        "question":question, 
        "answer":answer, 
        "answer_index":patient_records.index.values[0],
        "type":calculation_type, 
        "sub_type":subtraction_sub_type}
    vitals_counts.append(patient_records['vitals_count'].values[0])

print("vitals_counts: ", vitals_counts)
calc_1step_subtraction_df

vitals_counts:  [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(7), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(21), np.int64(22), np.int64(23), np.int64(32), np.int64(37), np.int64(40)]


,subject_id,context,question,answer,answer_index,type,sub_type
0,10511623,"{\n chart_time: 2140-10-05 22:40:00,\n a...",Calculate the pulse pressure at 2140-10-05 22:...,119,0,calculation,calc_1step_subtraction
1,18558304,"{\n chart_time: 2177-12-15 10:21:00,\n a...",Calculate the pulse pressure at 2177-12-15 10:...,62,0,calculation,calc_1step_subtraction
2,11227287,"{\n chart_time: 2173-04-13 16:41:00,\n a...",Calculate the pulse pressure at 2173-04-13 16:...,55,0,calculation,calc_1step_subtraction
3,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a...",Calculate the pulse pressure at 2135-06-15 11:...,58,0,calculation,calc_1step_subtraction
4,14411399,"{\n chart_time: 2135-04-02 22:53:00,\n a...",Calculate the pulse pressure at 2135-04-02 22:...,58,0,calculation,calc_1step_subtraction
5,14903260,"{\n chart_time: 2159-09-11 13:46:00,\n a...",Calculate the pulse pressure at 2159-09-11 13:...,75,0,calculation,calc_1step_subtraction
6,10407693,"{\n chart_time: 2127-02-18 04:02:00,\n a...",Calculate the pulse pressure at 2127-02-18 04:...,45,0,calculation,calc_1step_subtraction
7,13871299,"{\n chart_time: 2130-01-16 13:13:00,\n a...",Calculate the pulse pressure at 2130-01-16 16:...,42,1,calculation,calc_1step_subtraction
8,13067333,"{\n chart_time: 2177-08-23 17:51:00,\n a...",Calculate the pulse pressure at 2177-08-23 19:...,70,1,calculation,calc_1step_subtraction
9,10440642,"{\n chart_time: 2169-02-15 09:13:00,\n a...",Calculate the pulse pressure at 2169-06-01 11:...,67,1,calculation,calc_1step_subtraction


#### Multiplication - Temperature conversion factor

first step of Fahrenheit conversion

In [30]:
multiplication_sub_type = "calc_1step_multiplication"
question_template = "What is the value when multiplying the temperature at {charttime} by 1.8, round to 1 decimal places?"

calc_1step_multiplication_df = template_df.copy()
vitals_counts = []
for patient in df.groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(5, len(x)), random_state=RANDOM_STATE))['subject_id'].unique():
    patient_records = df[df['subject_id']==patient].reset_index(drop=True)
    patient_records = patient_records.loc[len(patient_records)//3:].sample(n=1)
    question = question_template.format(charttime=str(patient_records['charttime'].values[0]).split('.')[0].replace('T', ' '))
    answer = round(patient_records['temperature'].values[0] * 1.8, 1)
    calc_1step_multiplication_df.loc[len(calc_1step_multiplication_df)] = {
        "subject_id":patient, 
        "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
        "question":question, 
        "answer":answer, 
        "answer_index":patient_records.index.values[0],
        "type":calculation_type, 
        "sub_type":multiplication_sub_type}
    vitals_counts.append(patient_records['vitals_count'].values[0])

print("vitals_counts: ", vitals_counts)
calc_1step_multiplication_df

vitals_counts:  [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(7), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(21), np.int64(22), np.int64(23), np.int64(32), np.int64(37), np.int64(40)]


,subject_id,context,question,answer,answer_index,type,sub_type
0,10511623,"{\n chart_time: 2140-10-05 22:40:00,\n a...",What is the value when multiplying the tempera...,65.0,0,calculation,calc_1step_multiplication
1,18558304,"{\n chart_time: 2177-12-15 10:21:00,\n a...",What is the value when multiplying the tempera...,66.2,0,calculation,calc_1step_multiplication
2,11227287,"{\n chart_time: 2173-04-13 16:41:00,\n a...",What is the value when multiplying the tempera...,64.6,0,calculation,calc_1step_multiplication
3,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a...",What is the value when multiplying the tempera...,65.5,0,calculation,calc_1step_multiplication
4,14411399,"{\n chart_time: 2135-04-02 22:53:00,\n a...",What is the value when multiplying the tempera...,67.5,0,calculation,calc_1step_multiplication
5,14903260,"{\n chart_time: 2159-09-11 13:46:00,\n a...",What is the value when multiplying the tempera...,65.5,0,calculation,calc_1step_multiplication
6,10407693,"{\n chart_time: 2127-02-18 04:02:00,\n a...",What is the value when multiplying the tempera...,65.0,0,calculation,calc_1step_multiplication
7,13871299,"{\n chart_time: 2130-01-16 13:13:00,\n a...",What is the value when multiplying the tempera...,66.6,0,calculation,calc_1step_multiplication
8,13067333,"{\n chart_time: 2177-08-23 17:51:00,\n a...",What is the value when multiplying the tempera...,66.2,1,calculation,calc_1step_multiplication
9,10440642,"{\n chart_time: 2169-02-15 09:13:00,\n a...",What is the value when multiplying the tempera...,65.3,2,calculation,calc_1step_multiplication


#### Division - Shock Index

In [31]:
division_sub_type = "calc_1step_division"
question_template = "Calculate the Shock Index at {charttime}, round to 1 decimal places."

calc_1step_division_df = template_df.copy()
vitals_counts = []
for patient in df.groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(5, len(x)), random_state=RANDOM_STATE))['subject_id'].unique():
    patient_records = df[df['subject_id']==patient].reset_index(drop=True)
    patient_records = patient_records.loc[len(patient_records)//3:].sample(n=1)
    question = question_template.format(charttime=str(patient_records['charttime'].values[0]).split('.')[0].replace('T', ' '))
    answer = patient_records['heartrate'].values[0] / patient_records['sbp'].values[0]
    calc_1step_division_df.loc[len(calc_1step_division_df)] = {
        "subject_id":patient, 
        "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
        "question":question, 
        "answer":round(answer, 1), 
        "answer_index":patient_records.index.values[0],
        "type":calculation_type, 
        "sub_type":division_sub_type}
    vitals_counts.append(patient_records['vitals_count'].values[0])

print("vitals_counts: ", vitals_counts)
calc_1step_division_df

vitals_counts:  [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(7), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(21), np.int64(22), np.int64(23), np.int64(32), np.int64(37), np.int64(40)]


,subject_id,context,question,answer,answer_index,type,sub_type
0,10511623,"{\n chart_time: 2140-10-05 22:40:00,\n a...",Calculate the Shock Index at 2140-10-05 22:40:...,0.3,0,calculation,calc_1step_division
1,18558304,"{\n chart_time: 2177-12-15 10:21:00,\n a...",Calculate the Shock Index at 2177-12-15 10:21:...,0.6,0,calculation,calc_1step_division
2,11227287,"{\n chart_time: 2173-04-13 16:41:00,\n a...",Calculate the Shock Index at 2173-04-13 16:41:...,0.8,0,calculation,calc_1step_division
3,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a...",Calculate the Shock Index at 2135-06-15 11:11:...,0.5,0,calculation,calc_1step_division
4,14411399,"{\n chart_time: 2135-04-02 22:53:00,\n a...",Calculate the Shock Index at 2135-04-02 22:53:...,0.6,0,calculation,calc_1step_division
5,14903260,"{\n chart_time: 2159-09-11 13:46:00,\n a...",Calculate the Shock Index at 2159-09-11 13:46:...,0.8,0,calculation,calc_1step_division
6,10407693,"{\n chart_time: 2127-02-18 04:02:00,\n a...",Calculate the Shock Index at 2127-02-18 15:14:...,0.5,1,calculation,calc_1step_division
7,13871299,"{\n chart_time: 2130-01-16 13:13:00,\n a...",Calculate the Shock Index at 2130-01-16 13:13:...,0.9,0,calculation,calc_1step_division
8,13067333,"{\n chart_time: 2177-08-23 17:51:00,\n a...",Calculate the Shock Index at 2177-08-23 19:42:...,0.5,1,calculation,calc_1step_division
9,10440642,"{\n chart_time: 2169-02-15 09:13:00,\n a...",Calculate the Shock Index at 2169-02-15 09:13:...,0.6,2,calculation,calc_1step_division


#### merge

In [32]:
calc_1step_df = pd.concat([calc_1step_addition_df[:50], calc_1step_subtraction_df[:50], calc_1step_multiplication_df[:50], calc_1step_division_df[:50]], ignore_index=True)
calc_1step_df

,subject_id,context,question,answer,answer_index,type,sub_type
0,10511623,"{\n chart_time: 2140-10-05 22:40:00,\n a...",Calculate the sum of systolic and diastolic bl...,265.0,0,calculation,calc_1step_addition
1,18558304,"{\n chart_time: 2177-12-15 10:21:00,\n a...",Calculate the sum of systolic and diastolic bl...,180.0,0,calculation,calc_1step_addition
2,11227287,"{\n chart_time: 2173-04-13 16:41:00,\n a...",Calculate the sum of systolic and diastolic bl...,261.0,0,calculation,calc_1step_addition
3,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a...",Calculate the sum of systolic and diastolic bl...,218.0,0,calculation,calc_1step_addition
4,14411399,"{\n chart_time: 2135-04-02 22:53:00,\n a...",Calculate the sum of systolic and diastolic bl...,166.0,0,calculation,calc_1step_addition
...,...,...,...,...,...,...,...
195,13596929,"{\n chart_time: 2161-06-07 21:48:00,\n a...",Calculate the Shock Index at 2162-01-23 08:03:...,0.6,14,calculation,calc_1step_division
196,13663953,"{\n chart_time: 2184-07-15 01:39:00,\n a...",Calculate the Shock Index at 2187-02-03 09:25:...,0.5,14,calculation,calc_1step_division
197,19555898,"{\n chart_time: 2175-07-21 21:13:00,\n a...",Calculate the Shock Index at 2177-07-09 12:30:...,1.4,18,calculation,calc_1step_division
198,11834165,"{\n chart_time: 2136-07-17 00:45:00,\n a...",Calculate the Shock Index at 2139-07-17 09:05:...,0.9,20,calculation,calc_1step_division


In [33]:
calc_1step_df['sub_type'].value_counts()

sub_type
calc_1step_addition          50
calc_1step_subtraction       50
calc_1step_multiplication    50
calc_1step_division          50
Name: count, dtype: int64

#### Save

In [34]:
file_name = "calculation_1step.csv"

In [35]:
calc_1step_df.to_csv(os.path.join(calculation_output_dir, file_name), index=False)

In [36]:
formatted_calc_1step_df = format_df(calc_1step_df)
formatted_calc_1step_df.to_csv(os.path.join(formatted_calculation_output_dir, file_name), index=False)
formatted_calc_1step_df

,question,answer,open_ended_answer,dataset_source
0,"{\n chart_time: 2140-10-05 22:40:00,\n a...",265.0,265.0,calc_1step_addition
1,"{\n chart_time: 2177-12-15 10:21:00,\n a...",180.0,180.0,calc_1step_addition
2,"{\n chart_time: 2173-04-13 16:41:00,\n a...",261.0,261.0,calc_1step_addition
3,"{\n chart_time: 2135-06-15 11:11:00,\n a...",218.0,218.0,calc_1step_addition
4,"{\n chart_time: 2135-04-02 22:53:00,\n a...",166.0,166.0,calc_1step_addition
...,...,...,...,...
195,"{\n chart_time: 2161-06-07 21:48:00,\n a...",0.6,0.6,calc_1step_division
196,"{\n chart_time: 2184-07-15 01:39:00,\n a...",0.5,0.5,calc_1step_division
197,"{\n chart_time: 2175-07-21 21:13:00,\n a...",1.4,1.4,calc_1step_division
198,"{\n chart_time: 2136-07-17 00:45:00,\n a...",0.9,0.9,calc_1step_division


### 2-step

In [37]:
calc_2step_type = "calc_2step"

#### Addition + Division - average of respiratory rate in first/last 2 records

In [38]:
calc_2step_template1_type = f"{calc_2step_type}_addition_division"

question_template = "Calculate the average of respiratory rate at the {case} 2 records, round to 1 decimal places."
cases = ['first', 'last']

calc_2step_template1_df = template_df.copy()
vitals_counts = []
for case in cases:
    patients = df[df['vitals_count']>3].groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(3, len(x)), random_state=RANDOM_STATE))['subject_id'].unique()
    for patient in patients:
        patient_records = df[df['subject_id']==patient]
        question = question_template.format(case=case)
        answer = patient_records['resprate'].values[:2].mean().round(1) if case == 'first' else patient_records['resprate'].values[-2:].mean().round(1)
        calc_2step_template1_df.loc[len(calc_2step_template1_df)] = {
            "subject_id":patient, 
            "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
            "question":question, 
            "answer":round(answer, 1), 
            "answer_index":f"{case} 2 records",
            "type":calculation_type, 
            "sub_type":calc_2step_template1_type}
        vitals_counts.append(patient_records['vitals_count'].values[0])

print("vitals_counts: ", vitals_counts)
calc_2step_template1_df = calc_2step_template1_df[:50]
calc_2step_template1_df

vitals_counts:  [np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(10), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(21), np.int64(22), np.int64(23), np.int64(32), np.int64(37), np.int64(40), np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(10), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(21), np.int64(22), np.int64(23), np.int64(32), np.int64(37), np.int64(40)]


,subject_id,context,question,answer,answer_index,type,sub_type
0,10564151,"{\n chart_time: 2162-09-03 11:29:00,\n a...",Calculate the average of respiratory rate at t...,22.0,first 2 records,calculation,calc_2step_addition_division
1,13908377,"{\n chart_time: 2134-03-29 20:00:00,\n a...",Calculate the average of respiratory rate at t...,23.0,first 2 records,calculation,calc_2step_addition_division
2,11054342,"{\n chart_time: 2187-12-21 18:54:00,\n a...",Calculate the average of respiratory rate at t...,18.5,first 2 records,calculation,calc_2step_addition_division
3,11447083,"{\n chart_time: 2144-04-20 17:44:00,\n a...",Calculate the average of respiratory rate at t...,16.0,first 2 records,calculation,calc_2step_addition_division
4,13709749,"{\n chart_time: 2144-10-06 17:19:00,\n a...",Calculate the average of respiratory rate at t...,22.0,first 2 records,calculation,calc_2step_addition_division
5,11353832,"{\n chart_time: 2169-09-21 19:15:00,\n a...",Calculate the average of respiratory rate at t...,16.0,first 2 records,calculation,calc_2step_addition_division
6,17776188,"{\n chart_time: 2128-10-16 17:34:00,\n a...",Calculate the average of respiratory rate at t...,20.0,first 2 records,calculation,calc_2step_addition_division
7,10279832,"{\n chart_time: 2184-10-07 12:22:00,\n a...",Calculate the average of respiratory rate at t...,18.0,first 2 records,calculation,calc_2step_addition_division
8,16573945,"{\n chart_time: 2180-10-27 13:13:00,\n a...",Calculate the average of respiratory rate at t...,16.0,first 2 records,calculation,calc_2step_addition_division
9,14593165,"{\n chart_time: 2136-09-15 00:05:00,\n a...",Calculate the average of respiratory rate at t...,16.5,first 2 records,calculation,calc_2step_addition_division


In [39]:
calc_2step_template1_df['question'][0]

'Calculate the average of respiratory rate at the first 2 records, round to 1 decimal places.'

#### Subtraction + Division - a part of Mean Arterial Pressure (sbp-dbp)/3

In [40]:
calc_2step_template2_type = f"{calc_2step_type}_subtraction_division"
question_template = "Calculate a third of the pulse pressure at {charttime}, round to 1 decimal places."

calc_2step_template2_df = template_df.copy()
vitals_counts = []
for patient in df.groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(5, len(x)), random_state=RANDOM_STATE))['subject_id'].unique():
    patient_records = df[df['subject_id']==patient].reset_index(drop=True)
    patient_records = patient_records.loc[len(patient_records)//3:].sample(n=1)
    question = question_template.format(charttime=str(patient_records['charttime'].values[0]).split('.')[0].replace('T', ' '))
    answer = round((patient_records['sbp'].values[0] - patient_records['dbp'].values[0])/3, 1)
    calc_2step_template2_df.loc[len(calc_2step_template2_df)] = {
        "subject_id":patient, 
        "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
        "question":question, 
        "answer":answer, 
        "answer_index":patient_records.index.values[0],
        "type":calculation_type, 
        "sub_type":calc_2step_template2_type}
    vitals_counts.append(patient_records['vitals_count'].values[0])

print("vitals_counts: ", vitals_counts)
calc_2step_template2_df

vitals_counts:  [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(7), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(21), np.int64(22), np.int64(23), np.int64(32), np.int64(37), np.int64(40)]


,subject_id,context,question,answer,answer_index,type,sub_type
0,10511623,"{\n chart_time: 2140-10-05 22:40:00,\n a...",Calculate a third of the pulse pressure at 214...,39.7,0,calculation,calc_2step_subtraction_division
1,18558304,"{\n chart_time: 2177-12-15 10:21:00,\n a...",Calculate a third of the pulse pressure at 217...,20.7,0,calculation,calc_2step_subtraction_division
2,11227287,"{\n chart_time: 2173-04-13 16:41:00,\n a...",Calculate a third of the pulse pressure at 217...,18.3,0,calculation,calc_2step_subtraction_division
3,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a...",Calculate a third of the pulse pressure at 213...,19.3,0,calculation,calc_2step_subtraction_division
4,14411399,"{\n chart_time: 2135-04-02 22:53:00,\n a...",Calculate a third of the pulse pressure at 213...,19.3,0,calculation,calc_2step_subtraction_division
5,14903260,"{\n chart_time: 2159-09-11 13:46:00,\n a...",Calculate a third of the pulse pressure at 215...,25.0,0,calculation,calc_2step_subtraction_division
6,10407693,"{\n chart_time: 2127-02-18 04:02:00,\n a...",Calculate a third of the pulse pressure at 212...,15.0,0,calculation,calc_2step_subtraction_division
7,13871299,"{\n chart_time: 2130-01-16 13:13:00,\n a...",Calculate a third of the pulse pressure at 213...,11.0,0,calculation,calc_2step_subtraction_division
8,13067333,"{\n chart_time: 2177-08-23 17:51:00,\n a...",Calculate a third of the pulse pressure at 217...,23.0,0,calculation,calc_2step_subtraction_division
9,10440642,"{\n chart_time: 2169-02-15 09:13:00,\n a...",Calculate a third of the pulse pressure at 216...,17.7,2,calculation,calc_2step_subtraction_division


In [41]:
calc_2step_template2_df['question'][0]

'Calculate a third of the pulse pressure at 2140-10-05 22:40:00, round to 1 decimal places.'

#### Addition + Multiplication - Temperature to Fahrenheit

(°C × 1.8) + 32

In [42]:
calc_2step_template3_type = f"{calc_2step_type}_addition_multiplication"
question_template = "Convert temperature at {charttime} from Celsius to Fahrenheit, round to 1 decimal places."

calc_2step_template3_df = template_df.copy()
vitals_counts = []
for patient in df.groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(5, len(x)), random_state=RANDOM_STATE))['subject_id'].unique():
    patient_records = df[df['subject_id']==patient].reset_index(drop=True)
    patient_records = patient_records.loc[len(patient_records)//3:].sample(n=1)
    question = question_template.format(charttime=str(patient_records['charttime'].values[0]).split('.')[0].replace('T', ' '))
    answer = round((patient_records['temperature'].values[0] * 1.8 + 32), 1)
    calc_2step_template3_df.loc[len(calc_2step_template3_df)] = {
        "subject_id":patient, 
        "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
        "question":question, 
        "answer":answer, 
        "answer_index":patient_records.index.values[0],
        "type":calculation_type, 
        "sub_type":calc_2step_template3_type}
    vitals_counts.append(patient_records['vitals_count'].values[0])

print("vitals_counts: ", vitals_counts)
calc_2step_template3_df

vitals_counts:  [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(7), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(21), np.int64(22), np.int64(23), np.int64(32), np.int64(37), np.int64(40)]


,subject_id,context,question,answer,answer_index,type,sub_type
0,10511623,"{\n chart_time: 2140-10-05 22:40:00,\n a...",Convert temperature at 2140-10-05 22:40:00 fro...,97.0,0,calculation,calc_2step_addition_multiplication
1,18558304,"{\n chart_time: 2177-12-15 10:21:00,\n a...",Convert temperature at 2177-12-15 10:21:00 fro...,98.2,0,calculation,calc_2step_addition_multiplication
2,11227287,"{\n chart_time: 2173-04-13 16:41:00,\n a...",Convert temperature at 2173-04-13 16:41:00 fro...,96.6,0,calculation,calc_2step_addition_multiplication
3,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a...",Convert temperature at 2135-06-15 11:11:00 fro...,97.5,0,calculation,calc_2step_addition_multiplication
4,14411399,"{\n chart_time: 2135-04-02 22:53:00,\n a...",Convert temperature at 2135-04-02 22:53:00 fro...,99.5,0,calculation,calc_2step_addition_multiplication
5,14903260,"{\n chart_time: 2159-09-11 13:46:00,\n a...",Convert temperature at 2161-08-24 04:15:00 fro...,97.5,1,calculation,calc_2step_addition_multiplication
6,10407693,"{\n chart_time: 2127-02-18 04:02:00,\n a...",Convert temperature at 2127-02-18 15:14:00 fro...,97.7,1,calculation,calc_2step_addition_multiplication
7,13871299,"{\n chart_time: 2130-01-16 13:13:00,\n a...",Convert temperature at 2130-01-16 16:11:00 fro...,98.8,1,calculation,calc_2step_addition_multiplication
8,13067333,"{\n chart_time: 2177-08-23 17:51:00,\n a...",Convert temperature at 2177-08-23 17:51:00 fro...,97.7,0,calculation,calc_2step_addition_multiplication
9,10440642,"{\n chart_time: 2169-02-15 09:13:00,\n a...",Convert temperature at 2169-02-15 09:13:00 fro...,97.3,2,calculation,calc_2step_addition_multiplication


#### Subtraction + Multiplication - Percentage deviation

In [43]:
calc_2step_template4_type = f"{calc_2step_type}_subtraction_multiplication"
question_template = "Calculate the percentage that heart rate at {charttime} exceeds baseline (70 bpm), round to 1 decimal places."

calc_2step_template4_df = template_df.copy()
vitals_counts = []
for patient in df.groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(5, len(x)), random_state=RANDOM_STATE))['subject_id'].unique():
    patient_records = df[df['subject_id']==patient].reset_index(drop=True)
    patient_records = patient_records.loc[len(patient_records)//3:].sample(n=1)
    question = question_template.format(charttime=str(patient_records['charttime'].values[0]).split('.')[0].replace('T', ' '))
    answer = round((patient_records['heartrate'].values[0] - 70) / 70 * 100, 1)
    calc_2step_template4_df.loc[len(calc_2step_template4_df)] = {
        "subject_id":patient, 
        "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
        "question":question, 
        "answer":answer, 
        "answer_index":patient_records.index.values[0],
        "type":calculation_type, 
        "sub_type":calc_2step_template4_type}
    vitals_counts.append(patient_records['vitals_count'].values[0])

print("vitals_counts: ", vitals_counts)
calc_2step_template4_df

vitals_counts:  [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(7), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(21), np.int64(22), np.int64(23), np.int64(32), np.int64(37), np.int64(40)]


,subject_id,context,question,answer,answer_index,type,sub_type
0,10511623,"{\n chart_time: 2140-10-05 22:40:00,\n a...",Calculate the percentage that heart rate at 21...,-28.6,0,calculation,calc_2step_subtraction_multiplication
1,18558304,"{\n chart_time: 2177-12-15 10:21:00,\n a...",Calculate the percentage that heart rate at 21...,8.6,0,calculation,calc_2step_subtraction_multiplication
2,11227287,"{\n chart_time: 2173-04-13 16:41:00,\n a...",Calculate the percentage that heart rate at 21...,77.1,0,calculation,calc_2step_subtraction_multiplication
3,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a...",Calculate the percentage that heart rate at 21...,-10.0,0,calculation,calc_2step_subtraction_multiplication
4,14411399,"{\n chart_time: 2135-04-02 22:53:00,\n a...",Calculate the percentage that heart rate at 21...,2.9,0,calculation,calc_2step_subtraction_multiplication
5,14903260,"{\n chart_time: 2159-09-11 13:46:00,\n a...",Calculate the percentage that heart rate at 21...,-28.6,1,calculation,calc_2step_subtraction_multiplication
6,10407693,"{\n chart_time: 2127-02-18 04:02:00,\n a...",Calculate the percentage that heart rate at 21...,-5.7,0,calculation,calc_2step_subtraction_multiplication
7,13871299,"{\n chart_time: 2130-01-16 13:13:00,\n a...",Calculate the percentage that heart rate at 21...,38.6,0,calculation,calc_2step_subtraction_multiplication
8,13067333,"{\n chart_time: 2177-08-23 17:51:00,\n a...",Calculate the percentage that heart rate at 21...,11.4,1,calculation,calc_2step_subtraction_multiplication
9,10440642,"{\n chart_time: 2169-02-15 09:13:00,\n a...",Calculate the percentage that heart rate at 21...,42.9,1,calculation,calc_2step_subtraction_multiplication


#### merge

In [44]:
calc_2step_df = pd.concat([calc_2step_template1_df[:50], calc_2step_template2_df[:50], calc_2step_template3_df[:50], calc_2step_template4_df[:50]], ignore_index=True)
calc_2step_df

,subject_id,context,question,answer,answer_index,type,sub_type
0,10564151,"{\n chart_time: 2162-09-03 11:29:00,\n a...",Calculate the average of respiratory rate at t...,22.0,first 2 records,calculation,calc_2step_addition_division
1,13908377,"{\n chart_time: 2134-03-29 20:00:00,\n a...",Calculate the average of respiratory rate at t...,23.0,first 2 records,calculation,calc_2step_addition_division
2,11054342,"{\n chart_time: 2187-12-21 18:54:00,\n a...",Calculate the average of respiratory rate at t...,18.5,first 2 records,calculation,calc_2step_addition_division
3,11447083,"{\n chart_time: 2144-04-20 17:44:00,\n a...",Calculate the average of respiratory rate at t...,16.0,first 2 records,calculation,calc_2step_addition_division
4,13709749,"{\n chart_time: 2144-10-06 17:19:00,\n a...",Calculate the average of respiratory rate at t...,22.0,first 2 records,calculation,calc_2step_addition_division
...,...,...,...,...,...,...,...
195,13596929,"{\n chart_time: 2161-06-07 21:48:00,\n a...",Calculate the percentage that heart rate at 21...,-4.3,11,calculation,calc_2step_subtraction_multiplication
196,13663953,"{\n chart_time: 2184-07-15 01:39:00,\n a...",Calculate the percentage that heart rate at 21...,8.6,9,calculation,calc_2step_subtraction_multiplication
197,19555898,"{\n chart_time: 2175-07-21 21:13:00,\n a...",Calculate the percentage that heart rate at 21...,41.4,14,calculation,calc_2step_subtraction_multiplication
198,11834165,"{\n chart_time: 2136-07-17 00:45:00,\n a...",Calculate the percentage that heart rate at 21...,7.1,7,calculation,calc_2step_subtraction_multiplication


In [45]:
calc_2step_df['sub_type'].value_counts()

sub_type
calc_2step_addition_division             50
calc_2step_subtraction_division          50
calc_2step_addition_multiplication       50
calc_2step_subtraction_multiplication    50
Name: count, dtype: int64

#### save

In [46]:
file_name = "calculation_2step.csv"

In [47]:
calc_2step_df.to_csv(os.path.join(calculation_output_dir, file_name), index=False)

In [48]:
formatted_calc_2step_df = format_df(calc_2step_df)
formatted_calc_2step_df.to_csv(os.path.join(formatted_calculation_output_dir, file_name), index=False)
formatted_calc_2step_df

,question,answer,open_ended_answer,dataset_source
0,"{\n chart_time: 2162-09-03 11:29:00,\n a...",22.0,22.0,calc_2step_addition_division
1,"{\n chart_time: 2134-03-29 20:00:00,\n a...",23.0,23.0,calc_2step_addition_division
2,"{\n chart_time: 2187-12-21 18:54:00,\n a...",18.5,18.5,calc_2step_addition_division
3,"{\n chart_time: 2144-04-20 17:44:00,\n a...",16.0,16.0,calc_2step_addition_division
4,"{\n chart_time: 2144-10-06 17:19:00,\n a...",22.0,22.0,calc_2step_addition_division
...,...,...,...,...
195,"{\n chart_time: 2161-06-07 21:48:00,\n a...",-4.3,-4.3,calc_2step_subtraction_multiplication
196,"{\n chart_time: 2184-07-15 01:39:00,\n a...",8.6,8.6,calc_2step_subtraction_multiplication
197,"{\n chart_time: 2175-07-21 21:13:00,\n a...",41.4,41.4,calc_2step_subtraction_multiplication
198,"{\n chart_time: 2136-07-17 00:45:00,\n a...",7.1,7.1,calc_2step_subtraction_multiplication


### 3-steps

In [49]:
calc_3step_type = "calc_3step"

#### Subtraction + Division + Addition - Mean Arterial Pressure 
`(sbp-dbp)/3+dbp`

In [50]:
calc_3step_template1_type = f"{calc_3step_type}_map"
question_template = "Calculate the Mean Arterial Pressure at {charttime}, round to 1 decimal places."

calc_3step_template1_df = template_df.copy()
vitals_counts = []
for patient in df.groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(6, len(x)), random_state=RANDOM_STATE))['subject_id'].unique():
    patient_records = df[df['subject_id']==patient].reset_index(drop=True)
    patient_records_2 = patient_records.loc[len(patient_records)//3:].sample(n=min(2, len(patient_records)))
    for i in [0,1] if len(patient_records_2) > 1 else [0]:
        patient_records = patient_records_2.iloc[i:i+1]
        question = question_template.format(charttime=str(patient_records['charttime'].values[0]).split('.')[0].replace('T', ' '))
        answer = round((patient_records['sbp'].values[0] - patient_records['dbp'].values[0])/3 + patient_records['dbp'].values[0], 1)
        if len(calc_3step_template1_df) == 100:
            break
        calc_3step_template1_df.loc[len(calc_3step_template1_df)] = {
            "subject_id":patient, 
            "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
            "question":question, 
            "answer":answer, 
            "answer_index":patient_records.index.values[0],
            "type":calculation_type, 
            "sub_type":calc_3step_template1_type}
        vitals_counts.append(patient_records['vitals_count'].values[0])

print("vitals_counts: ", vitals_counts)
calc_3step_template1_df

vitals_counts:  [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(2), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(9), np.int64

,subject_id,context,question,answer,answer_index,type,sub_type
0,10511623,"{\n chart_time: 2140-10-05 22:40:00,\n a...",Calculate the Mean Arterial Pressure at 2140-1...,112.7,0,calculation,calc_3step_map
1,18558304,"{\n chart_time: 2177-12-15 10:21:00,\n a...",Calculate the Mean Arterial Pressure at 2177-1...,79.7,0,calculation,calc_3step_map
2,11227287,"{\n chart_time: 2173-04-13 16:41:00,\n a...",Calculate the Mean Arterial Pressure at 2173-0...,121.3,0,calculation,calc_3step_map
3,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a...",Calculate the Mean Arterial Pressure at 2135-0...,99.3,0,calculation,calc_3step_map
4,14411399,"{\n chart_time: 2135-04-02 22:53:00,\n a...",Calculate the Mean Arterial Pressure at 2135-0...,73.3,0,calculation,calc_3step_map
...,...,...,...,...,...,...,...
95,13663953,"{\n chart_time: 2184-07-15 01:39:00,\n a...",Calculate the Mean Arterial Pressure at 2190-0...,69.0,10,calculation,calc_3step_map
96,19555898,"{\n chart_time: 2175-07-21 21:13:00,\n a...",Calculate the Mean Arterial Pressure at 2175-0...,101.0,17,calculation,calc_3step_map
97,19555898,"{\n chart_time: 2175-07-21 21:13:00,\n a...",Calculate the Mean Arterial Pressure at 2177-1...,73.3,8,calculation,calc_3step_map
98,11834165,"{\n chart_time: 2136-07-17 00:45:00,\n a...",Calculate the Mean Arterial Pressure at 2140-0...,110.7,8,calculation,calc_3step_map


#### Subtraction + Division + Multiplication - Percentage deviation from baseline

In [51]:
calc_3step_template2_type = f"{calc_3step_type}_deviation_percentage"
question_template = "Calculate the percentage change in heart rate from {charttime_1}  to {charttime_2}, round to 2 decimal places."

calc_3step_template2_df = template_df.copy()
vitals_counts = []
for patient in df[df['vitals_count']>3].groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(12, len(x)), random_state=RANDOM_STATE))['subject_id'].unique()[:50]:
    patient_records = df[df['subject_id']==patient].reset_index(drop=True)
    patient_records_2 = patient_records.loc[min(len(patient_records)//3, len(patient_records)-2):].sample(n=min(3, len(patient_records))).sort_index()
    for i in [0,1] if len(patient_records_2) > 2 else [0]:
        patient_records = patient_records_2.iloc[i:i+2]
        question = question_template.format(charttime_1=str(patient_records['charttime'].values[0]).split('.')[0].replace('T', ' '), charttime_2=str(patient_records['charttime'].values[1]).split('.')[0].replace('T', ' '))
        answer = round((patient_records['heartrate'].values[1] - patient_records['heartrate'].values[0]) / patient_records['heartrate'].values[0] * 100, 2)
        calc_3step_template2_df.loc[len(calc_3step_template2_df)] = {
            "subject_id":patient, 
            "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
            "question":question, 
            "answer":answer, 
            "answer_index":f"{patient_records.index.values[0]}, {patient_records.index.values[1]}",
            "type":calculation_type, 
            "sub_type":calc_3step_template2_type}
        vitals_counts.append(patient_records['vitals_count'].values[0])

print("vitals_counts: ", vitals_counts)
calc_3step_template2_df

vitals_counts:  [np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.in

,subject_id,context,question,answer,answer_index,type,sub_type
0,10564151,"{\n chart_time: 2162-09-03 11:29:00,\n a...",Calculate the percentage change in heart rate ...,18.07,"1, 2",calculation,calc_3step_deviation_percentage
1,10564151,"{\n chart_time: 2162-09-03 11:29:00,\n a...",Calculate the percentage change in heart rate ...,-3.06,"2, 3",calculation,calc_3step_deviation_percentage
2,13908377,"{\n chart_time: 2134-03-29 20:00:00,\n a...",Calculate the percentage change in heart rate ...,6.80,"1, 2",calculation,calc_3step_deviation_percentage
3,13908377,"{\n chart_time: 2134-03-29 20:00:00,\n a...",Calculate the percentage change in heart rate ...,1.82,"2, 3",calculation,calc_3step_deviation_percentage
4,11054342,"{\n chart_time: 2187-12-21 18:54:00,\n a...",Calculate the percentage change in heart rate ...,9.41,"1, 2",calculation,calc_3step_deviation_percentage
...,...,...,...,...,...,...,...
95,12868753,"{\n chart_time: 2174-07-30 18:04:00,\n a...",Calculate the percentage change in heart rate ...,-16.35,"14, 17",calculation,calc_3step_deviation_percentage
96,17825043,"{\n chart_time: 2163-10-11 05:30:00,\n a...",Calculate the percentage change in heart rate ...,-16.25,"9, 14",calculation,calc_3step_deviation_percentage
97,17825043,"{\n chart_time: 2163-10-11 05:30:00,\n a...",Calculate the percentage change in heart rate ...,10.45,"14, 21",calculation,calc_3step_deviation_percentage
98,19200186,"{\n chart_time: 2142-09-07 09:58:00,\n a...",Calculate the percentage change in heart rate ...,-31.48,"10, 13",calculation,calc_3step_deviation_percentage


#### merge

In [52]:
calc_3step_df = pd.concat([calc_3step_template1_df, calc_3step_template2_df], ignore_index=True)
calc_3step_df

,subject_id,context,question,answer,answer_index,type,sub_type
0,10511623,"{\n chart_time: 2140-10-05 22:40:00,\n a...",Calculate the Mean Arterial Pressure at 2140-1...,112.70,0,calculation,calc_3step_map
1,18558304,"{\n chart_time: 2177-12-15 10:21:00,\n a...",Calculate the Mean Arterial Pressure at 2177-1...,79.70,0,calculation,calc_3step_map
2,11227287,"{\n chart_time: 2173-04-13 16:41:00,\n a...",Calculate the Mean Arterial Pressure at 2173-0...,121.30,0,calculation,calc_3step_map
3,10092020,"{\n chart_time: 2135-06-15 11:11:00,\n a...",Calculate the Mean Arterial Pressure at 2135-0...,99.30,0,calculation,calc_3step_map
4,14411399,"{\n chart_time: 2135-04-02 22:53:00,\n a...",Calculate the Mean Arterial Pressure at 2135-0...,73.30,0,calculation,calc_3step_map
...,...,...,...,...,...,...,...
195,12868753,"{\n chart_time: 2174-07-30 18:04:00,\n a...",Calculate the percentage change in heart rate ...,-16.35,"14, 17",calculation,calc_3step_deviation_percentage
196,17825043,"{\n chart_time: 2163-10-11 05:30:00,\n a...",Calculate the percentage change in heart rate ...,-16.25,"9, 14",calculation,calc_3step_deviation_percentage
197,17825043,"{\n chart_time: 2163-10-11 05:30:00,\n a...",Calculate the percentage change in heart rate ...,10.45,"14, 21",calculation,calc_3step_deviation_percentage
198,19200186,"{\n chart_time: 2142-09-07 09:58:00,\n a...",Calculate the percentage change in heart rate ...,-31.48,"10, 13",calculation,calc_3step_deviation_percentage


In [53]:
calc_3step_df['sub_type'].value_counts()

sub_type
calc_3step_map                     100
calc_3step_deviation_percentage    100
Name: count, dtype: int64

#### save

In [54]:
file_name = "calculation_3step.csv"

In [55]:
calc_3step_df.to_csv(os.path.join(calculation_output_dir, file_name), index=False)

In [56]:
formatted_calc_3step_df = format_df(calc_3step_df)
formatted_calc_3step_df.to_csv(os.path.join(formatted_calculation_output_dir, file_name), index=False)
formatted_calc_3step_df

,question,answer,open_ended_answer,dataset_source
0,"{\n chart_time: 2140-10-05 22:40:00,\n a...",112.70,112.70,calc_3step_map
1,"{\n chart_time: 2177-12-15 10:21:00,\n a...",79.70,79.70,calc_3step_map
2,"{\n chart_time: 2173-04-13 16:41:00,\n a...",121.30,121.30,calc_3step_map
3,"{\n chart_time: 2135-06-15 11:11:00,\n a...",99.30,99.30,calc_3step_map
4,"{\n chart_time: 2135-04-02 22:53:00,\n a...",73.30,73.30,calc_3step_map
...,...,...,...,...
195,"{\n chart_time: 2174-07-30 18:04:00,\n a...",-16.35,-16.35,calc_3step_deviation_percentage
196,"{\n chart_time: 2163-10-11 05:30:00,\n a...",-16.25,-16.25,calc_3step_deviation_percentage
197,"{\n chart_time: 2163-10-11 05:30:00,\n a...",10.45,10.45,calc_3step_deviation_percentage
198,"{\n chart_time: 2142-09-07 09:58:00,\n a...",-31.48,-31.48,calc_3step_deviation_percentage


## Comparison

In [57]:
comparison_type = "comparison"

In [58]:
comparison_output_dir = os.path.join(preprocess_mimic4ed_output_dir, "comparison")
formatted_comparison_output_dir = os.path.join(output_dir, "comparison")
os.makedirs(comparison_output_dir, exist_ok=True)
os.makedirs(formatted_comparison_output_dir, exist_ok=True)

### Quantifier 

#### 1 condition

##### first exceeded/less than/greater than

In [59]:
import random

comparison_comparative_type = f"{comparison_type}_comparative"
question_template = "What is the record datetime when {vital_parameter} first {comparator} {threshold}?"
vital_greater_than_threshold_mapping = {
    "sbp":120,
    "dbp":80,
    "heartrate":100,
    "resprate":20,
    "temperature":37.5,
}
vital_less_than_threshold_mapping = {
    "sbp":90,
    "dbp":60,
    "heartrate":60,
    "resprate":12,
    "temperature":36.5,
    "o2sat":95,
}

comparison_comparative_df = template_df.copy()
vitals_counts = []
# greater than
comparison_comparative_exceeded_type = f"{comparison_comparative_type}_exceeded"
for patient in df[df['vitals_count']>3].groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(4, len(x)), random_state=RANDOM_STATE))['subject_id'].unique():
    for vital_parameter in vital_greater_than_threshold_mapping:
        question = question_template.format(vital_parameter=vital_parameter, comparator=random.choice(["exceeded", "higher than"]), threshold=vital_greater_than_threshold_mapping[vital_parameter])
        patient_records = df[df['subject_id']==patient].reset_index(drop=True)
        answer_record = patient_records[patient_records[vital_parameter] > vital_greater_than_threshold_mapping[vital_parameter]]['charttime']
        if len(answer_record) > 0:
            answer = answer_record.values[0]
            comparison_comparative_df.loc[len(comparison_comparative_df)] = {
                "subject_id":patient, 
                "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
                "question":question, 
                "answer":answer, 
                "answer_index":answer_record.index.values[0],
                "type":comparison_type, 
                "sub_type":comparison_comparative_exceeded_type}
            vitals_counts.append(patient_records['vitals_count'].values[0])

# less than
comparison_comparative_drop_type = f"{comparison_comparative_type}_drop_below"
for patient in df[df['vitals_count']>3].groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(4, len(x)), random_state=RANDOM_STATE))['subject_id'].unique():
    for vital_parameter in vital_less_than_threshold_mapping:
        question = question_template.format(vital_parameter=vital_parameter, comparator=random.choice(["less than", "drop below"]), threshold=vital_less_than_threshold_mapping[vital_parameter])
        patient_records = df[df['subject_id']==patient].reset_index(drop=True)
        answer_record = patient_records[patient_records[vital_parameter] < vital_less_than_threshold_mapping[vital_parameter]]['charttime']
        if len(answer_record) > 0:
            answer = answer_record.values[0]
            comparison_comparative_df.loc[len(comparison_comparative_df)] = {
                    "subject_id":patient, 
                "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
                "question":question, 
                "answer":answer, 
                "answer_index":answer_record.index.values[0],
                "type":comparison_type, 
                "sub_type":comparison_comparative_drop_type}
            vitals_counts.append(patient_records['vitals_count'].values[0])


print("vitals_counts: ", vitals_counts)
comparison_comparative_df

vitals_counts:  [np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(6), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(11), np.int64(11), np.int64(11), np.int64(12), np.int64(13), np.int64(13), np.int64(

,subject_id,context,question,answer,answer_index,type,sub_type
0,10564151,"{\n chart_time: 2162-09-03 11:29:00,\n a...",What is the record datetime when sbp first hig...,2163-03-23 00:52:00,1,comparison,comparison_comparative_exceeded
1,10564151,"{\n chart_time: 2162-09-03 11:29:00,\n a...",What is the record datetime when resprate firs...,2163-03-22 19:49:00,0,comparison,comparison_comparative_exceeded
2,10564151,"{\n chart_time: 2162-09-03 11:29:00,\n a...",What is the record datetime when temperature f...,2163-03-22 19:49:00,0,comparison,comparison_comparative_exceeded
3,13908377,"{\n chart_time: 2134-03-29 20:00:00,\n a...",What is the record datetime when sbp first exc...,2134-03-29 20:00:00,0,comparison,comparison_comparative_exceeded
4,13908377,"{\n chart_time: 2134-03-29 20:00:00,\n a...",What is the record datetime when heartrate fir...,2134-03-29 20:00:00,0,comparison,comparison_comparative_exceeded
...,...,...,...,...,...,...,...
186,13634631,"{\n chart_time: 2177-10-17 01:04:00,\n a...",What is the record datetime when temperature f...,2181-08-28 10:29:00,6,comparison,comparison_comparative_drop_below
187,13634631,"{\n chart_time: 2177-10-17 01:04:00,\n a...",What is the record datetime when o2sat first d...,2181-11-14 02:04:00,16,comparison,comparison_comparative_drop_below
188,12407578,"{\n chart_time: 2116-12-29 13:36:00,\n a...",What is the record datetime when heartrate fir...,2119-04-18 17:16:00,35,comparison,comparison_comparative_drop_below
189,12407578,"{\n chart_time: 2116-12-29 13:36:00,\n a...",What is the record datetime when temperature f...,2117-04-02 17:02:00,6,comparison,comparison_comparative_drop_below


###### save

In [60]:
file_name = "comparative.csv"

In [61]:
comparison_comparative_df.to_csv(os.path.join(comparison_output_dir, file_name), index=False)

In [62]:
formatted_comparison_comparative_df = format_df(comparison_comparative_df)
formatted_comparison_comparative_df.to_csv(os.path.join(formatted_comparison_output_dir, file_name), index=False)
formatted_comparison_comparative_df

,question,answer,open_ended_answer,dataset_source
0,"{\n chart_time: 2162-09-03 11:29:00,\n a...",2163-03-23 00:52:00,2163-03-23 00:52:00,comparison_comparative_exceeded
1,"{\n chart_time: 2162-09-03 11:29:00,\n a...",2163-03-22 19:49:00,2163-03-22 19:49:00,comparison_comparative_exceeded
2,"{\n chart_time: 2162-09-03 11:29:00,\n a...",2163-03-22 19:49:00,2163-03-22 19:49:00,comparison_comparative_exceeded
3,"{\n chart_time: 2134-03-29 20:00:00,\n a...",2134-03-29 20:00:00,2134-03-29 20:00:00,comparison_comparative_exceeded
4,"{\n chart_time: 2134-03-29 20:00:00,\n a...",2134-03-29 20:00:00,2134-03-29 20:00:00,comparison_comparative_exceeded
...,...,...,...,...
186,"{\n chart_time: 2177-10-17 01:04:00,\n a...",2181-08-28 10:29:00,2181-08-28 10:29:00,comparison_comparative_drop_below
187,"{\n chart_time: 2177-10-17 01:04:00,\n a...",2181-11-14 02:04:00,2181-11-14 02:04:00,comparison_comparative_drop_below
188,"{\n chart_time: 2116-12-29 13:36:00,\n a...",2119-04-18 17:16:00,2119-04-18 17:16:00,comparison_comparative_drop_below
189,"{\n chart_time: 2116-12-29 13:36:00,\n a...",2117-04-02 17:02:00,2117-04-02 17:02:00,comparison_comparative_drop_below


##### superlative

In [63]:
comparison_superlative_type = f"{comparison_type}_superlative"

question_template = "What is the record datetime when {vital_parameter} is {comparator}?"
vital_parameters = ["temperature", "heartrate", "sbp", "dbp", "resprate", "o2sat"]
comparators = ["highest", "lowest"]

comparison_superlative_df = template_df.copy()
vitals_counts = []
# greater than
for patient in df[df['vitals_count']>6].groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(1, len(x)), random_state=RANDOM_STATE))['subject_id'].unique():
    for vital_parameter in vital_parameters:
        for comparator in comparators:
            comparison_superlative_comparator_type = f"{comparison_superlative_type}_{comparator}"
            question = question_template.format(vital_parameter=vital_parameter, comparator=comparator)
            patient_record = df[df['subject_id']==patient].reset_index(drop=True).sort_values(by=vital_parameter, ascending=comparator == "lowest")
            answer = patient_record["charttime"].values[0]
            comparison_superlative_df.loc[len(comparison_superlative_df)] = {
                "subject_id":patient, 
                "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
                "question":question, 
                "answer":answer, 
                "answer_index":patient_record.iloc[0].name,
                "type":comparison_type, 
                "sub_type":comparison_superlative_comparator_type}
            vitals_counts.append(patient_record['vitals_count'].values[0])

print("vitals_counts: ", vitals_counts)
comparison_superlative_df

vitals_counts:  [np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(7), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(8), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(9), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(10), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(11), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(12), np.int64(13

,subject_id,context,question,answer,answer_index,type,sub_type
0,14593165,"{\n chart_time: 2136-09-15 00:05:00,\n a...",What is the record datetime when temperature i...,2138-06-10 20:56:00,3,comparison,comparison_superlative_highest
1,14593165,"{\n chart_time: 2136-09-15 00:05:00,\n a...",What is the record datetime when temperature i...,2139-09-19 17:04:00,2,comparison,comparison_superlative_lowest
2,14593165,"{\n chart_time: 2136-09-15 00:05:00,\n a...",What is the record datetime when heartrate is ...,2137-12-11 10:51:00,6,comparison,comparison_superlative_highest
3,14593165,"{\n chart_time: 2136-09-15 00:05:00,\n a...",What is the record datetime when heartrate is ...,2139-09-19 17:04:00,2,comparison,comparison_superlative_lowest
4,14593165,"{\n chart_time: 2136-09-15 00:05:00,\n a...",What is the record datetime when sbp is highest?,2137-12-11 10:51:00,6,comparison,comparison_superlative_highest
...,...,...,...,...,...,...,...
187,12407578,"{\n chart_time: 2116-12-29 13:36:00,\n a...",What is the record datetime when dbp is lowest?,2116-12-31 16:53:00,0,comparison,comparison_superlative_lowest
188,12407578,"{\n chart_time: 2116-12-29 13:36:00,\n a...",What is the record datetime when resprate is h...,2119-09-24 05:00:00,18,comparison,comparison_superlative_highest
189,12407578,"{\n chart_time: 2116-12-29 13:36:00,\n a...",What is the record datetime when resprate is l...,2119-09-20 02:15:00,14,comparison,comparison_superlative_lowest
190,12407578,"{\n chart_time: 2116-12-29 13:36:00,\n a...",What is the record datetime when o2sat is high...,2116-12-31 16:53:00,0,comparison,comparison_superlative_highest


###### save

In [64]:
file_name = "superlative.csv"

In [65]:
comparison_superlative_df.to_csv(os.path.join(comparison_output_dir, file_name), index=False)

In [66]:
formatted_comparison_superlative_df = format_df(comparison_superlative_df)
formatted_comparison_superlative_df.to_csv(os.path.join(formatted_comparison_output_dir, file_name), index=False)
formatted_comparison_superlative_df

,question,answer,open_ended_answer,dataset_source
0,"{\n chart_time: 2136-09-15 00:05:00,\n a...",2138-06-10 20:56:00,2138-06-10 20:56:00,comparison_superlative_highest
1,"{\n chart_time: 2136-09-15 00:05:00,\n a...",2139-09-19 17:04:00,2139-09-19 17:04:00,comparison_superlative_lowest
2,"{\n chart_time: 2136-09-15 00:05:00,\n a...",2137-12-11 10:51:00,2137-12-11 10:51:00,comparison_superlative_highest
3,"{\n chart_time: 2136-09-15 00:05:00,\n a...",2139-09-19 17:04:00,2139-09-19 17:04:00,comparison_superlative_lowest
4,"{\n chart_time: 2136-09-15 00:05:00,\n a...",2137-12-11 10:51:00,2137-12-11 10:51:00,comparison_superlative_highest
...,...,...,...,...
187,"{\n chart_time: 2116-12-29 13:36:00,\n a...",2116-12-31 16:53:00,2116-12-31 16:53:00,comparison_superlative_lowest
188,"{\n chart_time: 2116-12-29 13:36:00,\n a...",2119-09-24 05:00:00,2119-09-24 05:00:00,comparison_superlative_highest
189,"{\n chart_time: 2116-12-29 13:36:00,\n a...",2119-09-20 02:15:00,2119-09-20 02:15:00,comparison_superlative_lowest
190,"{\n chart_time: 2116-12-29 13:36:00,\n a...",2116-12-31 16:53:00,2116-12-31 16:53:00,comparison_superlative_highest


In [67]:
comparison_df = pd.concat([comparison_superlative_df,comparison_comparative_df], ignore_index=True)
comparison_df['sub_type'].value_counts()

sub_type
comparison_comparative_exceeded      115
comparison_superlative_highest        96
comparison_superlative_lowest         96
comparison_comparative_drop_below     76
Name: count, dtype: int64

## Summary

In [68]:
summary_type = "summary"

In [69]:
summary_output_dir = os.path.join(preprocess_mimic4ed_output_dir, "summary")
formatted_summary_output_dir = os.path.join(output_dir, "summary")
os.makedirs(summary_output_dir, exist_ok=True)
os.makedirs(formatted_summary_output_dir, exist_ok=True)

### Count with thres

In [70]:
question_template = "How many time the patient has {issue}? Return the number of records."
issues = {
    "Tachycardia":{"heartrate":100}, 
    "Mean Arterial Pressure higher than 100 mmHg": {"map":100},
    "Shock Index higher than 0.7": {"shock_index":0.7}
}

summary_df = template_df.copy()
vitals_counts = []
df['map'] = round((df['sbp'] - df['dbp']) / 3 + df['dbp'], 1)
df['shock_index'] = round(df['heartrate'] / df['sbp'], 1)
for patient in df[df['vitals_count']>3].groupby(["vitals_count"], group_keys=False).apply(lambda x: x.sample(n=min(50, len(x)), random_state=RANDOM_STATE))['subject_id'].unique():
    for issue in issues:
        summary_issue_type = f"summary_count_by_{issue.lower().replace(' ', '.')}"
        
        patient_records = df[df['subject_id']==patient].reset_index(drop=True)
        filtered_records = patient_records[patient_records['heartrate'] > 100 if issue == "Tachycardia" else patient_records['map'] > 100 if issue == "Mean Arterial Pressure higher than 100 mmHg" else patient_records['shock_index'] > 0.7]
        question = question_template.format(issue=issue)
        answer = filtered_records.shape[0]
        summary_df.loc[len(summary_df)] = {
            "subject_id":patient, 
            "context":df_verbal[df_verbal['subject_id']==patient]['standard_verbal'].values[0], 
            "question":question, 
            "answer":answer, 
            "answer_index":", ".join(filtered_records.index.values.astype(str)) if len(filtered_records) > 0 else '',
            "type":summary_type,    
            "sub_type":summary_issue_type}
        vitals_counts.append(patient_records['vitals_count'].values[0])
            

print("vitals_counts: ", vitals_counts)
summary_df



vitals_counts:  [np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(5), np.int64(6), np.int64(6), np.int64(6), np.int64

,subject_id,context,question,answer,answer_index,type,sub_type
0,10564151,"{\n chart_time: 2162-09-03 11:29:00,\n a...",How many time the patient has Tachycardia? Ret...,0,,summary,summary_count_by_tachycardia
1,10564151,"{\n chart_time: 2162-09-03 11:29:00,\n a...",How many time the patient has Mean Arterial Pr...,0,,summary,summary_count_by_mean.arterial.pressure.higher...
2,10564151,"{\n chart_time: 2162-09-03 11:29:00,\n a...",How many time the patient has Shock Index high...,1,0,summary,summary_count_by_shock.index.higher.than.0.7
3,13908377,"{\n chart_time: 2134-03-29 20:00:00,\n a...",How many time the patient has Tachycardia? Ret...,4,"0, 1, 2, 3",summary,summary_count_by_tachycardia
4,13908377,"{\n chart_time: 2134-03-29 20:00:00,\n a...",How many time the patient has Mean Arterial Pr...,0,,summary,summary_count_by_mean.arterial.pressure.higher...
...,...,...,...,...,...,...,...
193,13634631,"{\n chart_time: 2177-10-17 01:04:00,\n a...",How many time the patient has Mean Arterial Pr...,13,"6, 8, 9, 12, 13, 14, 17, 18, 21, 30, 31, 33, 35",summary,summary_count_by_mean.arterial.pressure.higher...
194,13634631,"{\n chart_time: 2177-10-17 01:04:00,\n a...",How many time the patient has Shock Index high...,9,"4, 5, 16, 20, 29, 32, 33, 34, 36",summary,summary_count_by_shock.index.higher.than.0.7
195,12407578,"{\n chart_time: 2116-12-29 13:36:00,\n a...",How many time the patient has Tachycardia? Ret...,24,"2, 6, 7, 8, 9, 10, 11, 12, 13, 15, 17, 18, 19,...",summary,summary_count_by_tachycardia
196,12407578,"{\n chart_time: 2116-12-29 13:36:00,\n a...",How many time the patient has Mean Arterial Pr...,36,"1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15,...",summary,summary_count_by_mean.arterial.pressure.higher...


#### merge

In [71]:
summary_df['sub_type'].value_counts()

sub_type
summary_count_by_tachycardia                                    66
summary_count_by_mean.arterial.pressure.higher.than.100.mmhg    66
summary_count_by_shock.index.higher.than.0.7                    66
Name: count, dtype: int64

#### save

In [72]:
file_name = "summary.csv"

In [73]:
summary_df.to_csv(os.path.join(summary_output_dir, file_name), index=False)

In [74]:
formatted_summary_df = format_df(summary_df)
formatted_summary_df.to_csv(os.path.join(formatted_summary_output_dir, file_name), index=False)
formatted_summary_df

,question,answer,open_ended_answer,dataset_source
0,"{\n chart_time: 2162-09-03 11:29:00,\n a...",0,0,summary_count_by_tachycardia
1,"{\n chart_time: 2162-09-03 11:29:00,\n a...",0,0,summary_count_by_mean.arterial.pressure.higher...
2,"{\n chart_time: 2162-09-03 11:29:00,\n a...",1,1,summary_count_by_shock.index.higher.than.0.7
3,"{\n chart_time: 2134-03-29 20:00:00,\n a...",4,4,summary_count_by_tachycardia
4,"{\n chart_time: 2134-03-29 20:00:00,\n a...",0,0,summary_count_by_mean.arterial.pressure.higher...
...,...,...,...,...
193,"{\n chart_time: 2177-10-17 01:04:00,\n a...",13,13,summary_count_by_mean.arterial.pressure.higher...
194,"{\n chart_time: 2177-10-17 01:04:00,\n a...",9,9,summary_count_by_shock.index.higher.than.0.7
195,"{\n chart_time: 2116-12-29 13:36:00,\n a...",24,24,summary_count_by_tachycardia
196,"{\n chart_time: 2116-12-29 13:36:00,\n a...",36,36,summary_count_by_mean.arterial.pressure.higher...


# Statistic

In [75]:
statistic_output_path = os.path.join(preprocess_mimic4ed_output_dir, "statistic.log")

statistics = []
for task in ["retrieval", "calculation", "comparison", "summary"]:
    for file in os.listdir(os.path.join(output_dir, task)):
        data = pd.read_csv(os.path.join(output_dir, task, file))
        statistics.append({
            "task": task,
            "sub_type": file.split(".")[0],
            "num_records": len(data)
        })

statistic_df = pd.DataFrame(statistics)
with open(statistic_output_path, "w") as f:
    markdown_table = statistic_df.to_markdown()
    f.write(markdown_table)
    f.write("\n")
    f.write("\n")
    f.write("Total number of records: " + str(statistic_df['num_records'].sum()))
    print(markdown_table)
    print("Total number of records: " + str(statistic_df['num_records'].sum()))

|    | task        | sub_type               |   num_records |
|---:|:------------|:-----------------------|--------------:|
|  0 | retrieval   | direct_retrieval       |           240 |
|  1 | calculation | calculation_1step      |           200 |
|  2 | calculation | calculation_2step      |           200 |
|  3 | calculation | calculation_3step      |           200 |
|  4 | comparison  | comparison_superlative |           192 |
|  5 | comparison  | comparison_comparative |           191 |
|  6 | comparison  | superlative            |           192 |
|  7 | comparison  | comparative            |           191 |
|  8 | summary     | summary                |           198 |
Total number of records: 1804
